# Comparison of semantics
In this notebook we compare the different semantics on a few examples. We show how `EagerQualitative` and `Rosi` can report violations and satisfactions early, while the delayed semantics always wait until the end of the interval to produce a verdict. We also show how `RoSI` can provide more information about the robustness of the formula at each timestamp, even before it can be definitively classified as satisfied or violated.

In [4]:
import ostl_python as ostl

# The four semantics supported
SEMANTICS = [
    "DelayedQualitative",
    "EagerQualitative",
    "DelayedQuantitative",
    "Rosi",
]

print("Loaded ostl_python. Semantics:")
for s in SEMANTICS:
    print("-", s)

Loaded ostl_python. Semantics:
- DelayedQualitative
- EagerQualitative
- DelayedQuantitative
- Rosi


How this notebook works:
1) Choose one STL formula
2) Run the same trace with each semantics
3) Compare verdict type and when verdicts appear

Core API used in every section:
- phi = ostl.parse_formula('...')
- monitor = ostl.Monitor(phi, semantics='...')
- out = monitor.update(signal, value, timestamp)
- out.verdicts()

## 1) Same formula, different value types

Formula: `x > 0.5`

This is the easiest case (no temporal operator):
- both qualitative semantics return `bool`
- delayed quantitative returns `float`
- RoSI returns interval `(lower bound, upper bound)`

In [5]:
formula = "x > 0.5"
trace = [
    ("x", 0.2, 0.0),
    ("x", 0.6, 1.0),
    ("x", 0.4, 2.0),
]

for semantics in SEMANTICS:
    phi = ostl.parse_formula(formula)
    monitor = ostl.Monitor(phi, semantics=semantics)

    print("\n" + "=" * 70)
    print(f"Semantics: {semantics}")

    first_value = None

    for signal, value, t in trace:
        out = monitor.update(signal, value, t)
        verdicts = out.verdicts()
        print(f"update({signal}, {value}, {t}) -> {verdicts}")


Semantics: DelayedQualitative
update(x, 0.2, 0.0) -> [(0.0, False)]
update(x, 0.6, 1.0) -> [(1.0, True)]
update(x, 0.4, 2.0) -> [(2.0, False)]

Semantics: EagerQualitative
update(x, 0.2, 0.0) -> [(0.0, False)]
update(x, 0.6, 1.0) -> [(1.0, True)]
update(x, 0.4, 2.0) -> [(2.0, False)]

Semantics: DelayedQuantitative
update(x, 0.2, 0.0) -> [(0.0, -0.3)]
update(x, 0.6, 1.0) -> [(1.0, 0.09999999999999998)]
update(x, 0.4, 2.0) -> [(2.0, -0.09999999999999998)]

Semantics: Rosi
update(x, 0.2, 0.0) -> [(0.0, (-0.3, -0.3))]
update(x, 0.6, 1.0) -> [(1.0, (0.09999999999999998, 0.09999999999999998))]
update(x, 0.4, 2.0) -> [(2.0, (-0.09999999999999998, -0.09999999999999998))]


The output from ```out.verdicts()``` is a list of tuples of the form `(timestamp, verdict)`, where `verdict` is the value of the formula at that timestamp according to the semantics used. There may be multiple verdicts at the same timestamp.

We see also how the different semantics agree, e.g. when the formula is violated the qualitative semantics return `False`, the delayed quantitative semantics return a negative value, and the RoSI semantics return an interval that does not include zero.

## 2) Early violation: `G[0,2](x > 0.5)`

`G` means "always".
If one bad sample appears early, `EagerQualitative` and `Rosi` can report violation early.
Delayed semantics always wait until the end of the interval to produce a verdict, so they report violation at timestamp 2.

In [9]:
formula = "G[0, 2](x > 0.5)"
trace = [
    ("x", 0.9, 0.0),
    ("x", 0.1, 0.5),  # early bad sample
    ("x", 0.8, 1.0),
    ("x", 0.9, 2.0),
    ("x", 0.9, 3.0),
]

for semantics in SEMANTICS:
    phi = ostl.parse_formula(formula)
    monitor = ostl.Monitor(phi, semantics=semantics)

    print("\n" + "=" * 70)
    print(f"Semantics: {semantics}")

    first_emission_time = None

    for signal, value, t in trace:
        out = monitor.update(signal, value, t)
        verdicts = out.verdicts()
        print(f"update({signal}, {value}, {t}) -> {verdicts}")


Semantics: DelayedQualitative
update(x, 0.9, 0.0) -> []
update(x, 0.1, 0.5) -> []
update(x, 0.8, 1.0) -> []
update(x, 0.9, 2.0) -> [(0.0, False)]
update(x, 0.9, 3.0) -> [(0.5, False), (1.0, True)]

Semantics: EagerQualitative
update(x, 0.9, 0.0) -> []
update(x, 0.1, 0.5) -> [(0.0, False), (0.5, False)]
update(x, 0.8, 1.0) -> []
update(x, 0.9, 2.0) -> []
update(x, 0.9, 3.0) -> [(1.0, True)]

Semantics: DelayedQuantitative
update(x, 0.9, 0.0) -> []
update(x, 0.1, 0.5) -> []
update(x, 0.8, 1.0) -> []
update(x, 0.9, 2.0) -> [(0.0, -0.4)]
update(x, 0.9, 3.0) -> [(0.5, -0.4), (1.0, 0.30000000000000004)]

Semantics: Rosi
update(x, 0.9, 0.0) -> [(0.0, (-inf, 0.4))]
update(x, 0.1, 0.5) -> [(0.0, (-inf, -0.4)), (0.5, (-inf, -0.4))]
update(x, 0.8, 1.0) -> [(0.0, (-inf, -0.4)), (0.5, (-inf, -0.4)), (1.0, (-inf, 0.30000000000000004))]
update(x, 0.9, 2.0) -> [(0.0, (-0.4, -0.4)), (0.5, (-inf, -0.4)), (1.0, (-inf, 0.30000000000000004)), (2.0, (-inf, 0.4))]
update(x, 0.9, 3.0) -> [(0.5, (-0.4, -0.4))

We observe that at $t=2s$, all semantics agree that the formula is violated for $t=0s$, however ```EagerQualitative``` and ```Rosi``` can report this violation already at $t=0.5s$ where the delayed semantics have not yet produced any verdicts. They are also able to report a violation at $t=1s$ at this time. 

## 3) Early satisfaction: `F[0,2](x > 0.5)`

`F` means "eventually".
If one good sample appears early, `EagerQualitative` and `Rosi` can report satisfaction early. `RoSI` shows this with the lower bound of the interval becoming definitively positive. Delayed semantics always wait until the end of the interval to produce a verdict, so they report satisfaction at timestamp 2.

In [10]:
formula = "F[0, 2](x > 0.5)"
trace = [
    ("x", 0.1, 0.0),
    ("x", 0.7, 0.6),  # early good sample
    ("x", 0.2, 1.2),
    ("x", 0.1, 2.1),
    ("x", 0.1, 3.0),
]

for semantics in SEMANTICS:
    phi = ostl.parse_formula(formula)
    monitor = ostl.Monitor(phi, semantics=semantics)

    print("\n" + "=" * 70)
    print(f"Semantics: {semantics}")

    first_emission_time = None

    for signal, value, t in trace:
        out = monitor.update(signal, value, t)
        verdicts = out.verdicts()
        print(f"update({signal}, {value}, {t}) -> {verdicts}")


Semantics: DelayedQualitative
update(x, 0.1, 0.0) -> []
update(x, 0.7, 0.6) -> []
update(x, 0.2, 1.2) -> []
update(x, 0.1, 2.1) -> [(0.0, True)]
update(x, 0.1, 3.0) -> [(0.6, True)]

Semantics: EagerQualitative
update(x, 0.1, 0.0) -> []
update(x, 0.7, 0.6) -> [(0.0, True), (0.6, True)]
update(x, 0.2, 1.2) -> []
update(x, 0.1, 2.1) -> []
update(x, 0.1, 3.0) -> []

Semantics: DelayedQuantitative
update(x, 0.1, 0.0) -> []
update(x, 0.7, 0.6) -> []
update(x, 0.2, 1.2) -> []
update(x, 0.1, 2.1) -> [(0.0, 0.19999999999999996)]
update(x, 0.1, 3.0) -> [(0.6, 0.19999999999999996)]

Semantics: Rosi
update(x, 0.1, 0.0) -> [(0.0, (-0.4, inf))]
update(x, 0.7, 0.6) -> [(0.0, (0.19999999999999996, inf)), (0.6, (0.19999999999999996, inf))]
update(x, 0.2, 1.2) -> [(0.0, (0.19999999999999996, inf)), (0.6, (0.19999999999999996, inf)), (1.2, (-0.3, inf))]
update(x, 0.1, 2.1) -> [(0.0, (0.19999999999999996, 0.19999999999999996)), (0.6, (0.19999999999999996, inf)), (1.2, (-0.3, inf)), (2.1, (-0.4, inf))]
u

Once again we observe that all semantics agree with `EagerQualitative` and `Rosi` reporting satisfaction at $t=0.7s$ where the delayed semantics have not yet produced any verdicts. `RoSI` shows this with the lower bound of the interval becoming definitively positive at this time.